# 経済データ分析Ⅱ　第6回
# ダミー変数と交差項

第2回では、持ち家の床面積を人口密度で説明した。住宅・土地統計調査には借家の床面積もあり、第2回の冒頭で見たように、石川県と東京都の差は持ち家では61.3 m²、借家では7.9 m²だった。今回は、持ち家と借家の2つを1つの回帰式で扱い、次の2つを調べる。

- 同じ人口密度の県で、持ち家は借家より平均して何 m² 広いか。
- 人口密度が高いほど住宅が狭いという関係（傾き）は、持ち家と借家で違うか。

「持ち家か借家か」のような質的な情報は、0と1をとる変数で回帰式に入れる。

## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。途中のセルから実行して「変数が見つからない」というエラー（`NameError`）が出たら、先頭のセルから実行し直してください。演習の記入欄はダブルクリックすると編集でき、`Shift + Enter` で表示に戻ります。

JupyterLite では **Python (Pyodide)**、ローカルの Jupyter では Python カーネルを使います。データは Notebook に収録しているので、CSV ファイルの準備は要りません。授業の終わりには Notebook を保存し、ブラウザで使った場合は `.ipynb` をダウンロードして手元にも残しましょう。

## 1. 準備

次のセルで、今回使うライブラリを読み込む。`japanize_matplotlib_jlite` は、図の文字を日本語で表示するためのパッケージで、経済データ分析Ⅰでも使った。ブラウザでは `piplite.install()` でこのパッケージを入れる。`if sys.platform == 'emscripten':` から始まる部分は、ブラウザ（JupyterLite）で実行しているときに必要なライブラリを用意する処理である。`np`、`pd`、`plt` は第2回からと同じ略称である。

`statsmodels.formula.api` は、回帰式を文字で書いて推定する機能で、`smf` という略称で読み込む。今回からはこの書き方を使う。

In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels'])
    import piplite
    await piplite.install('japanize-matplotlib-jlite')

from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite
import statsmodels.formula.api as smf

## 2. 持ち家と借家の延べ面積

### 2.1. データの読み込み

総務省統計局[『統計でみる都道府県のすがた2026』](https://www.stat.go.jp/data/k-sugata/naiyou.html)の47都道府県の値を、第2回と同じ方法で読み込む。人口密度は2024年、住宅の1住宅当たりの延べ面積は2023年の値である。下のセルは、文字で書いた表を `StringIO()` で読み込める形にし、`pd.read_csv()` で表にする。

In [ ]:
housing_csv = """prefecture,density,owned_area,rented_area
北海道,64.3,120.1,50.9
青森県,120.8,145.2,52.5
岩手県,75.0,145.8,49.3
宮城県,308.7,127.9,45.4
秋田県,77.1,154.3,51.3
山形県,108.4,159.9,50.0
福島県,126.4,139.2,48.9
茨城県,460.1,126.9,47.5
栃木県,294.2,128.7,47.6
群馬県,297.1,128.7,48.8
埼玉県,1930.6,106.4,44.4
千葉県,1212.3,109.0,44.3
東京都,6444.7,90.5,39.8
神奈川県,3817.4,97.9,42.1
新潟県,166.8,155.7,48.3
富山県,234.7,167.5,49.0
石川県,262.3,151.8,47.7
福井県,176.3,163.5,51.9
山梨県,177.1,132.6,48.8
長野県,146.5,144.1,49.9
岐阜県,180.4,141.9,50.8
静岡県,453.5,127.5,47.8
愛知県,1442.0,123.7,46.5
三重県,296.3,131.1,49.5
滋賀県,349.0,137.2,47.4
京都府,546.4,110.5,41.5
大阪府,4596.0,99.9,43.2
兵庫県,635.3,114.2,47.6
奈良県,348.1,127.8,53.6
和歌山県,186.3,123.8,52.1
鳥取県,151.4,147.2,51.1
島根県,95.7,151.1,51.4
岡山県,257.4,131.4,46.9
広島県,320.1,119.5,46.0
山口県,209.6,124.6,50.5
徳島県,165.2,133.8,50.0
香川県,488.6,132.1,49.5
愛媛県,224.8,123.7,50.6
高知県,92.4,117.3,51.3
福岡県,1020.9,114.8,45.9
佐賀県,322.9,138.9,53.7
長崎県,303.1,121.5,51.4
熊本県,229.0,125.0,50.1
大分県,171.1,123.5,47.6
宮崎県,133.6,115.2,51.0
鹿児島県,166.8,108.8,50.7
沖縄県,642.4,106.0,48.8"""
wide = pd.read_csv(StringIO(housing_csv))
wide.head()

`density` は総面積1km²当たりの人口密度（人/km²）、`owned_area` は持ち家、`rented_area` は借家の延べ面積（m²）である。1行が1都道府県で、持ち家と借家の面積が横に並んでいる。

### 2.2. 持ち家と借家の差

都道府県ごとに、持ち家の面積から借家の面積を引いた差を `gap` の列に入れる。`.sort_values('gap', ascending=False)` は、差の大きい順に行を並べ替える。`ascending=False` は大きい順の指定で、省くと小さい順になる。`.head()` で先頭の5行を表示する。

In [ ]:
wide['gap'] = wide['owned_area'] - wide['rented_area']
wide.sort_values('gap', ascending=False).head()

差が最も大きいのは富山県の118.5 m²で、福井県、山形県、新潟県、石川県が続く。どれも日本海側の県である。`.tail()` は末尾の5行を表示するので、差の小さい5県が分かる。

In [ ]:
wide.sort_values('gap', ascending=False).tail()

差が最も小さいのは東京都の50.7 m²で、神奈川県、大阪府、沖縄県、鹿児島県が続く。2つの表の `owned_area` の列を見ると、差の大きい5県の持ち家はどれも150 m²を超え、差の小さい5県の持ち家はどれも110 m²より狭い。`rented_area` の列は、どちらの表でも約40〜52 m²である。差の大きさの違いは、主に持ち家の面積の違いから来ている。

47都道府県の差の平均を求めておく。

In [ ]:
print(f"差の平均：{wide['gap'].mean():.2f} m²")

### 2.3. 人口密度と延べ面積の散布図

横軸を人口密度の自然対数、縦軸を延べ面積にして、持ち家と借家の点を同じ図に描く。まず、`np.log()` で人口密度の自然対数を求める。

In [ ]:
log_density_wide = np.log(wide['density'])
log_density_wide.head().round(2)

北海道の人口密度64.3人/km²の自然対数は4.16である。次のセルで散布図を描く。

- `fig, ax = plt.subplots()` で図を用意する。`fig` は図全体、`ax` は点を描く場所である。
- `ax.scatter(横軸, 縦軸)` で点を描く。2回使うと、2つのグループの点が別の色になる。`s=15` は点の大きさ、`label` は凡例に出す名前である。
- `ax.set()` で横軸（`xlabel`）と縦軸（`ylabel`）の名前を付け、`ax.legend()` で凡例を表示する。
- 最後の `plt.show()` で図を表示する。

In [ ]:
fig, ax = plt.subplots()
ax.scatter(log_density_wide, wide['owned_area'], s=15, label='持ち家')
ax.scatter(log_density_wide, wide['rented_area'], s=15, label='借家')
ax.set(xlabel='人口密度の自然対数', ylabel='延べ面積（m²）')
ax.legend()
plt.show()

上の点の集まりが持ち家、下が借家である。持ち家は人口密度の高い県ほどはっきり狭くなるが、借家の点はほぼ横に並んでいる。持ち家と借家では、面積の水準も、人口密度との関係の強さも違う。第3節では水準の違いを、第5節では関係の強さの違いを、回帰式で表す。

### 2.4. 持ち家と借家を縦に並べた表

持ち家と借家を1つの回帰式で扱うため、1行が「1都道府県の持ち家」または「1都道府県の借家」になるように並べ直す。まず持ち家の表を作る。`pd.DataFrame({'列の名前': 中身, ...})` は、列の名前と中身を組にして表を作る。`area`（延べ面積）には持ち家の面積を入れる。`'owned': 1` のように1つの数を書いた列は、すべての行が同じ値になる。`owned` は、持ち家なら1、借家なら0にする列である。

In [ ]:
owned_rows = pd.DataFrame({'prefecture': wide['prefecture'], 'density': wide['density'],
                           'area': wide['owned_area'], 'owned': 1})
owned_rows.head()

同じように借家の表を作る。`area` には借家の面積を入れ、`owned` はすべての行で0にする。

In [ ]:
rented_rows = pd.DataFrame({'prefecture': wide['prefecture'], 'density': wide['density'],
                            'area': wide['rented_area'], 'owned': 0})
rented_rows.head()

`pd.concat([表1, 表2])` は、2つの表を縦につなぐ。`ignore_index=True` は、つないだ後の行番号を0から付け直す指定である。`len()` で行数を数える。

In [ ]:
housing = pd.concat([owned_rows, rented_rows], ignore_index=True)
print(f'行数：{len(housing)}')

47都道府県の持ち家と借家で、94行の表ができた。回帰に使う人口密度の自然対数の列を加え、`.iloc[[0, 1, 47, 48]]` で0、1、47、48番目の行を取り出して表示する。

In [ ]:
housing['log_density'] = np.log(housing['density'])
housing.iloc[[0, 1, 47, 48]].round(2)

0行目と47行目はどちらも北海道で、0行目が持ち家（`owned` が1）、47行目が借家（`owned` が0）である。人口密度は同じ値になっている。

`owned` のように0と1だけをとる変数を**ダミー変数**という。

## 3. ダミー変数を使った回帰

### 3.1. ダミー変数だけの回帰

まず、延べ面積をダミー変数 `owned` だけに回帰する。

$$面積_i=\beta_0+\delta_0\,owned_i+u_i$$

`smf.ols('area ~ owned', data=housing)` は、`~` の左の列を被説明変数、右の列を説明変数とする回帰式を作る。切片は自動で入るので、`sm.add_constant()` は要らない。`.fit()` で推定する。

In [ ]:
fit1 = smf.ols('area ~ owned', data=housing).fit()
print(fit1.params.round(2))

切片は48.61、`owned` の係数は80.49である。借家（$owned_i=0$）の当てはめ値は $\beta_0$ だけ、持ち家（$owned_i=1$）の当てはめ値は $\beta_0+\delta_0$ になる。持ち家と借家の平均を求めて比べる。`housing.groupby('owned')['area'].mean()` は、`owned` の値（0と1）ごとに延べ面積の平均を求める。

In [ ]:
housing.groupby('owned')['area'].mean().round(2)

借家の平均48.61 m²が切片に、持ち家の平均129.10 m²と借家の平均の差 $129.10-48.61=80.49$ m²が `owned` の係数に一致する。2.2節で求めた都道府県ごとの差の平均とも同じ値である。ダミー変数の係数は、もう一方のグループ（ここでは借家）との平均の差を表す。0になる側のグループを**基準のグループ**という。

### 3.2. 人口密度を加えた回帰

人口密度の対数を説明変数に加える。`+` で説明変数をつなぐ。

$$面積_i=\beta_0+\delta_0\,owned_i+\beta_1\,\ln(人口密度_i)+u_i$$

In [ ]:
fit2 = smf.ols('area ~ owned + log_density', data=housing).fit()
print(fit2.params.round(2))

`owned` の係数は80.49で、人口密度が同じ県の持ち家は借家より平均80.49 m²広い。この式では、持ち家の切片は $\beta_0+\delta_0$、借家の切片は $\beta_0$ で、人口密度の傾き −6.76 はどちらも同じである。ダミー変数は切片だけをグループごとに変え、2本の平行な直線を当てはめる。

このデータでは、持ち家と借家で人口密度の値がまったく同じなので、`owned` の係数は3.1節と同じ値になった。

当てはめた式を図で確かめる。2.3節の散布図に、持ち家と借家の当てはめ直線を加える。まず、`housing[housing['owned'] == 1]` で `owned` が1の行（持ち家）だけを取り出す。借家の行も同じように取り出す。

In [ ]:
owned_part = housing[housing['owned'] == 1]
rented_part = housing[housing['owned'] == 0]
print(f'持ち家：{len(owned_part)}行、借家：{len(rented_part)}行')

次に、直線を引く横軸の点を用意する。`np.linspace(最小, 最大, 100)` は、最小から最大までに100個の点を等しい間隔で並べる。ここでは、人口密度の対数の最小と最大を使う。

In [ ]:
grid = np.linspace(housing['log_density'].min(), housing['log_density'].max(), 100)
print(f'最初の点：{grid[0]:.2f}、最後の点：{grid[-1]:.2f}、点の数：{len(grid)}')

それぞれの点での直線の高さを、`fit2.params` の係数を式に代入して計算する。借家の直線の高さは $\beta_0+\beta_1\ln(人口密度)$、持ち家の直線の高さはこれに $\delta_0$ を足した値である。

In [ ]:
b2 = fit2.params
rented_line2 = b2['Intercept'] + b2['log_density'] * grid
owned_line2 = rented_line2 + b2['owned']
print(f'左端の高さ：持ち家 {owned_line2[0]:.1f}、借家 {rented_line2[0]:.1f}')

人口密度の対数が最も小さい左端では、持ち家の直線の高さが139.6 m²、借家が59.1 m²で、その差は `owned` の係数80.49 m²にあたる。2.3節と同じ散布図に、`ax.plot(横軸, 縦軸)` で直線を加える。`ax.plot()` は点を線でつないで描くので、100個の点の高さをつなぐと直線になる。

In [ ]:
fig, ax = plt.subplots()
ax.scatter(owned_part['log_density'], owned_part['area'], s=15, label='持ち家')
ax.scatter(rented_part['log_density'], rented_part['area'], s=15, label='借家')
ax.plot(grid, owned_line2)
ax.plot(grid, rented_line2)
ax.set(xlabel='人口密度の自然対数', ylabel='延べ面積（m²）')
ax.legend()
plt.show()

2本の直線は平行で、間隔はどこでも80.49 m²である。持ち家の点は直線より急な右下がりに並び、人口密度の低い県では直線より上に、高い県では下に来やすい。借家の点はその反対で、人口密度の低い県では直線より下に、高い県では上に来る。傾きを持ち家と借家で同じにした式では、2.3節の図で見た関係の強さの違いを表せない。

## 4. 人口密度によるグループ分け

### 4.1. 条件から作るダミー変数

ダミー変数は、数値の列から条件で作ることもできる。人口密度が1,000人/km²を超えるかどうかで、都道府県を2つのグループに分ける。`housing['density'] > 1000` は、行ごとに条件を満たすかどうかを True と False で表す。`.astype(int)` は、True を1、False を0に直す。

In [ ]:
housing['high_density'] = (housing['density'] > 1000).astype(int)
housing.iloc[[9, 10, 11, 12]][['prefecture', 'density', 'high_density']]

群馬県（297.1人/km²）は0、埼玉県、千葉県、東京都は1になった。0と1の列を `.sum()` で足すと、1の行の数が分かる。

In [ ]:
print(f"high_density が1の行：{housing['high_density'].sum()}")

1の行は14行で、持ち家と借家で2行ずつあるので、7都府県にあたる。`high_density` を説明変数に加えて回帰する。

In [ ]:
fit_high = smf.ols('area ~ owned + high_density', data=housing).fit()
print(fit_high.params.round(2))

`high_density` の係数 −16.42 は、持ち家か借家かが同じなら、人口密度が1,000人/km²を超える県の面積はそれ以外の県より平均16.42 m²狭いことを表す。基準のグループは、`high_density` が0の40道府県である。

### 4.2. 数値をいくつかの区分に分ける

4.1節の2つのグループをさらに分け、人口密度を「低い」「中くらい」「高い」の3つの区分にする。ここでは説明のために、300人/km²以下を「低い」、300人より多く1,000人以下を「中くらい」、1,000人より多い県を「高い」とする。「高い」は、4.1節で `high_density` を1にした7都府県である。

`pd.cut(列, bins=[境目], labels=[名前])` は、数値を境目で区切って区分の名前を付ける。`bins` の隣り合う2つの数が1つの区分の範囲になり、範囲には右端の値を含む。

In [ ]:
housing['density_group'] = pd.cut(housing['density'], bins=[0, 300, 1000, 10000],
                                  labels=['低い', '中くらい', '高い'])
housing.iloc[[0, 3, 10]][['prefecture', 'density', 'density_group']]

北海道（64.3人/km²）は「低い」、宮城県（308.7人/km²）は「中くらい」、埼玉県（1,930.6人/km²）は「高い」になった。`.groupby('density_group', observed=True)['prefecture'].count()` は、区分ごとに行の数を数える。`observed=True` は、行のある区分だけを数える指定である。

In [ ]:
housing.groupby('density_group', observed=True)['prefecture'].count()

区分ごとの行数は、低い56行、中くらい24行、高い14行である。持ち家と借家で1都道府県が2行ずつあるので、都道府県の数はその半分の28、12、7になる。

このように、いくつかの値のどれか1つをとる変数を**カテゴリー変数**という。

### 4.3. カテゴリー変数を使った回帰

区分が3つあるとき、ダミー変数は3−1＝2個使う。1つの区分を基準のグループにして回帰式から外し、残りの区分ごとに「その区分なら1」のダミー変数を置く。3つとも入れると、3つのダミー変数の和がどの行でも1になり、切片の列と同じになって、第3回の仮定3（完全な多重共線性がない）が成り立たない。

回帰式の中で `C(列)` と書くと、statsmodels がこのダミー変数を作る。基準のグループは、最初の区分（ここでは「低い」）になる。

In [ ]:
fit3 = smf.ols('area ~ owned + C(density_group)', data=housing).fit()
print(fit3.params.round(2))

`C(density_group)[T.中くらい]` の係数 −7.21 は、持ち家か借家かが同じなら、人口密度が「中くらい」の県の面積は「低い」県より平均7.21 m²狭いことを表す。「高い」県は「低い」県より平均18.58 m²狭い。

基準のグループは `C(列, Treatment('区分の名前'))` で替えられる。「高い」を基準にする。

In [ ]:
fit3b = smf.ols("area ~ owned + C(density_group, Treatment('高い'))", data=housing).fit()
print(fit3b.params.round(2))

今度は「低い」県が「高い」県より18.58 m²広く、「中くらい」の県が11.37 m²広い。$-7.21-(-18.58)=11.37$ なので、比べる相手が変わっただけで、区分どうしの差は同じである。`owned` の係数も変わらない。

## 5. 交差項

### 5.1. 傾きがグループで違う回帰式

3.2節の式では、人口密度の傾きが持ち家と借家で同じだった。傾きもグループごとに変えるには、ダミー変数と人口密度の積を説明変数に加える。この積を**交差項**という。

$$面積_i=\beta_0+\delta_0\,owned_i+\beta_1\,\ln(人口密度_i)+\delta_1\,owned_i\times\ln(人口密度_i)+u_i$$

借家（$owned_i=0$）では交差項が0になり、傾きは $\beta_1$ である。持ち家（$owned_i=1$）では傾きが $\beta_1+\delta_1$ になる。$\delta_1$ は、持ち家と借家の傾きの差を表す。

回帰式の中で `owned * log_density` と書くと、`owned`、`log_density`、交差項 `owned:log_density` の3つが説明変数に入る。

In [ ]:
fit4 = smf.ols('area ~ owned * log_density', data=housing).fit()
print(fit4.params.round(3))

借家の傾きは $\beta_1=-2.266$、持ち家の傾きは $\beta_1+\delta_1=-2.266-8.978=-11.244$ である。持ち家の傾きは、第2回で持ち家の床面積を人口密度の対数に回帰した傾き −11.24 と同じになる。交差項を入れた式は、持ち家と借家で別々に回帰したときと同じ直線を当てはめるからである。

人口密度が2倍の県では、借家は約 $2.27\times\ln2\approx1.6$ m²、持ち家は約 $11.24\times\ln2\approx7.8$ m²狭い。人口密度による広さの違いは、持ち家のほうがずっと大きい。

この式の `owned` の係数131.851は、人口密度の自然対数が0（人口密度が1人/km²）の県での、持ち家と借家の差を表す。このデータの人口密度の対数は4.2から8.8の間にあるので、この係数を持ち家と借家の差としてそのまま読むことはできない。交差項を入れた式では、持ち家と借家の差は人口密度によって変わる。演習1の問3で計算しよう。

### 5.2. 傾きの違う2本の直線

3.2節と同じ方法で、交差項を入れた式の当てはめ直線を描く。まず直線の高さを計算する。借家の傾きは $\beta_1$、持ち家の傾きは $\beta_1+\delta_1$ なので、持ち家の直線には `b4['log_density'] + b4['owned:log_density']` を使う。

In [ ]:
b4 = fit4.params
rented_line4 = b4['Intercept'] + b4['log_density'] * grid
owned_line4 = b4['Intercept'] + b4['owned'] + (b4['log_density'] + b4['owned:log_density']) * grid
print(f'左端の高さ：持ち家 {owned_line4[0]:.1f}、借家 {rented_line4[0]:.1f}')

左端では、持ち家の直線の高さが146.6 m²で3.2節の139.6 m²より高く、借家は52.1 m²で3.2節の59.1 m²より低い。3.2節の図と同じ散布図に、この2本の直線を描く。

In [ ]:
fig, ax = plt.subplots()
ax.scatter(owned_part['log_density'], owned_part['area'], s=15, label='持ち家')
ax.scatter(rented_part['log_density'], rented_part['area'], s=15, label='借家')
ax.plot(grid, owned_line4)
ax.plot(grid, rented_line4)
ax.set(xlabel='人口密度の自然対数', ylabel='延べ面積（m²）')
ax.legend()
plt.show()

3.2節の図と比べると、持ち家の直線は右下がりが急になり、借家の直線はほぼ水平になった。どちらの直線も、点の並びに沿っている。

## 6. 結果の読み方

人口密度が同じ県で比べると、持ち家は借家より平均80.49 m²広い（3.2節）。人口密度が高いほど住宅が狭いという関係は持ち家で強く、借家ではわずかである（5.1節）。

ダミー変数は、グループの間で切片を変える。交差項は、グループの間で傾きを変える。グループの違いが平均の差だけか、説明変数との関係にも及ぶかは、交差項の係数で分かる。

## 演習1：ダミー変数と交差項の係数を使った計算

1. 3.2節の `fit2` の係数を使い、人口密度の自然対数が6（人口密度は約400人/km²）の県について、持ち家と借家の延べ面積の当てはめ値をそれぞれ計算せよ。2つの差は何 m² か。
2. 4.3節の `fit3` の係数を使い、人口密度が「高い」県の借家の当てはめ値を計算せよ。その値は、`fit3b`（「高い」を基準にした回帰）のどの係数と同じか。
3. 5.1節の `fit4` の係数を使い、人口密度の自然対数が4.5の県と8.5の県のそれぞれで、持ち家と借家の当てはめ値の差（持ち家 − 借家）を計算せよ。問1の差と比べ、差が人口密度によって変わる理由を、交差項の働きから2〜4文で説明せよ。

追加問題：`smf.ols('area ~ owned * C(density_group)', data=housing)` で、`owned` と人口密度の区分の交差項を入れた回帰を推定せよ。人口密度の区分ごとに持ち家と借家の差を係数から計算し、差が最も大きい区分と最も小さい区分を答えよ。

問1〜問3は計算欄で係数を使って計算する。追加問題では、4.3節と5.1節のセルを参考にしてコードを書く。

In [ ]:
# 問1〜問3の計算欄
# 係数は fit2.params['owned'] のように名前で取り出せる。
# 切片の名前は 'Intercept'、交差項の名前は 'owned:log_density' である。

In [ ]:
# 追加問題の計算欄

### 演習1の記入欄

- 問1：持ち家と借家の当てはめ値、その差：
- 問2：「高い」県の借家の当てはめ値と、同じ値になる係数：
- 問3：人口密度の対数が4.5と8.5のときの差、差が変わる理由（2〜4文）：
- 追加問題：区分ごとの差、差が最も大きい区分と最も小さい区分：

<details>
<summary>演習1のヒント</summary>

問1の借家の当てはめ値は、`fit2.params['Intercept'] + fit2.params['log_density'] * 6` で計算できる。持ち家の当てはめ値は、これに `fit2.params['owned']` を足す。

問2の「高い」県の借家は、`owned` が0で、「高い」のダミー変数が1である。係数の名前は、`print(fit3.params)` の表示どおりに `'C(density_group)[T.高い]'` と書く。

問3では、持ち家の式から借家の式を引くと、差は $\delta_0+\delta_1\times\ln(人口密度)$ になる。3.2節の式には交差項が無いので、差はどの人口密度でも `owned` の係数のままである。

追加問題では、「低い」区分の持ち家と借家の差が `owned` の係数になる。「中くらい」と「高い」の区分の差は、`owned` の係数にそれぞれの交差項の係数を足して求める。

</details>

## 参考文献

この回の構成は下の本の第7章（7.1節、7.3〜7.5節）に沿っている。この本は著者のサイトで無料で読める。

- [Heiss, F. and Brunner, D. (2024) Using Python for Introductory Econometrics, 2nd edition](https://www.urfie.net/downloads/PDF/UPfIE_web.pdf)

## 関数の資料

- [statsmodels：式による回帰（`smf.ols`）](https://www.statsmodels.org/stable/example_formulas.html)
- [pandas：`pd.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- [pandas：`pd.cut`](https://pandas.pydata.org/docs/reference/api/pandas.cut.html)